# Value - the building block of a Neuron

In [ ]:
import math

In [ ]:
from graphviz import Digraph


def trace(root):
    nodes, edges = set(), set()

    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)

    build(root)
    return nodes, edges


def draw_dot(root, format="svg", rankdir="LR"):
    """
    format: png | svg | ...
    rankdir: TB (top to bottom graph) | LR (left to right)
    """
    assert rankdir in ["LR", "TB"]
    nodes, edges = trace(root)
    dot = Digraph(
        format=format, graph_attr={"rankdir": rankdir}
    )  # , node_attr={'rankdir': 'TB'})

    for n in nodes:
        dot.node(
            name=str(id(n)),
            label="{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad),
            shape="record",
        )
        if n._op:
            dot.node(name=str(id(n)) + n._op, label=n._op)
            dot.edge(str(id(n)) + n._op, str(id(n)))

    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

In [ ]:
class Value:
    """Value is the single scalar node in a computational graph."""

    def __init__(self, data, _children=(), _op="", label=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self.label = label
        self._prev = _children
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data})"

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad = out.grad * other.data
            other.grad = out.grad * self.data

        out._backward = _backward

        return out

    def __rmul__(self, other):  # other * self
        return self * other

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), _op="+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward

        return out

    def __radd__(self, other):  # other + self
        return self + other

    def tahn(self):
        # Hyperbolic tangent definition: (exp(2*x) - 1) / (exp(2*x) + 1)
        t = (math.exp(2 * self.data) - 1) / (math.exp(2 * self.data) + 1)
        out = Value(t, (self,), "tahn")

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward

        return out

    def backward(self):
        """Calculate gradients for each Value object in a graph in topological order starting from the last one."""
        topo = []
        visited = set()

        def build_topo(node):
            if node not in visited:
                visited.add(node)  # add node to seen nodes
                for child in node._prev:
                    build_topo(child)
                topo.append(node)  # append not to the end of topo

        build_topo(self)

        # initialize grad (this is the right most Value object on which backward is called)
        self.grad = 1.0
        # calculate gradients in reversed order, starting with last
        for node in reversed(topo):
            node._backward()

    def zero_grad(self):
        self.grad = 0.0


# tests()

In [ ]:
a2 = Value(2.0, label="a")
b2 = Value(3.0, label='b')
c2 = a2 + b2; c2.label = "c"
o2 = c2.tahn()
o2.label = "o"
o2.backward()
draw_dot(o2)

In [ ]:
a1 = Value(2.0, label="a")
b1 = Value(3.0, label="b")
e1 = a1 * b1
e1.label = "e"
n1 = e1 + e1
n1.label = "n"
o1 = n1.tahn()
o1.label = "o"
o1.backward()
draw_dot(o1)

In [ ]:
def tests():
    a_value = 2.0
    b_value = 3.0
    a_label = "a"
    b_label = "b"
    value_op = "op"
    a = Value(a_value)
    b = Value(b_value, label=b_label)
    init_grad = 1.0

    # init
    assert a.data == a_value
    assert a.label == ""  # label is optional, initially is empty string
    a.label = a_label
    assert a.label == a_label
    a._op = value_op
    assert a._op == value_op
    a._prev = ()
    assert a._prev == ()
    a._prev = (a, a)
    assert isinstance(a._prev, tuple)
    assert a._prev == (a, a)
    assert repr(a) == f"Value(data={a_value})"

    # multiplication
    o = a * a  # can multiply 2 Value objects
    assert o.data == a_value * a_value
    o = a * 3  # can multiply a Value object to an int or float
    assert o.data == a_value * 3
    assert isinstance(o.data, float)
    o = a * 3.0
    assert o.data == a_value * 3.0
    assert isinstance(o.data, float)
    o = 3 * a  # can multiply in or float to Value object
    o = 3 * a
    assert o.data == a_value * 3
    assert isinstance(o.data, float)
    o = 3.0 * a
    assert o.data == a_value * 3.0
    assert isinstance(o.data, float)

    # addition
    o = a + a  # can add two Value objects
    assert o.data == a_value + a_value
    o = a + 3  # can add Value object to an int or float
    assert o.data == a_value + 3
    o = a + 3.0
    assert o.data == a_value + 3.0
    assert isinstance(o.data, float)
    o = 3 + a  # can add an int or float to a Value object
    assert o.data == a_value + 3
    assert isinstance(o.data, float)
    o = 3.0 + a
    assert o.data == a_value + 3.0
    assert isinstance(o.data, float)

    # tahn
    o = a.tahn()
    assert o.data == math.tanh(a.data)

    # backward
    o = a
    o.backward()
    ## a single value has grad initiated to 1.0
    assert o.grad == init_grad
    ## an addition grad (to itself)
    ## addition grad definition: a.grad = o.grad * 1
    a.zero_grad()  # discard gradient before any new forward pass
    o = a + a  # forward pass
    o.backward()  # backward pass
    assert o.grad == init_grad
    assert a.grad == (o.grad * 1.0) + (o.grad * 1.0)

    ## addition grad (to other Value)
    ## addition grad definition: a.grad = o.grad * 1; b.grad = o.grad * 1
    a.zero_grad()  # discard gradient before any new forward pass
    b.zero_grad()
    o = a + b  # forward pass
    o.backward()  # backward pass
    assert o.grad == init_grad
    assert a.grad == o.grad * 1.0
    assert b.grad == o.grad * 1.0

    ## multiplication grad
    ## multiplication grad definition: a.grad = o.grad * b.grad; b.grad = o.grad * a.grad
    a.zero_grad()  # discard gradient before any new forward pass
    b.zero_grad()
    o = a * b  # forward pass
    o.backward()  # backward pass
    assert o.grad == init_grad
    assert a.grad == o.grad * b.data
    assert b.grad == o.grad * a.data

    ## tahn grad
    ## tahn grad definition: tahn.d = tahn_derivative * o.grad
    a.zero_grad()  # discard gradient before any new forward pass
    b.zero_grad()
    c = a + b
    o = c.tahn()
    o.backward()
    # Hyperbolic tangent definition: (exp(2*x) - 1) / (exp(2*x) + 1)
    tahn_value = (math.exp(2 * c.data) - 1) / (math.exp(2 * c.data) + 1)
    tahn_derivative = 1 - tahn_value**2
    assert o.grad == init_grad
    assert c.grad == tahn_derivative * o.grad


tests()

In [ ]:
# Building the Value

x1 = Value(2.0, label="x1")
w1 = Value(0.4, label="w1")
x2 = Value(0.0, label="x2")
w2 = Value(3.0, label="w2")
b1 = Value(5.0, label="b")

x1w1 = x1 * w1
x1w1.label = "x1w1"
x2w2 = x2 * w2
x2w2.label = "x2w2"

x1w1x2w2 = x1w1 + x2w2
x1w1x2w2.label = "x1w1x2w2"

n = x1w1x2w2 + b1
n.label = "n"

o1 = n.tahn()
o1.label = "o"

draw_dot(o1)

In [ ]:
# Backward propagation
o1.backward()

draw_dot(o1)